# Bài 6 trang 26: Giảm số màu của ảnh bằng k-Means

**Họ tên:** Lê Nguyễn Anh Du  
**MSSV:** 6451071011

Gom các pixel RGB thành 32 cụm màu, thay mỗi pixel bằng màu của tâm cụm, hiển thị ảnh trước và sau, rồi lưu ảnh kết quả. Nếu đã có `dog.png` trong thư mục làm việc của Colab, notebook sẽ dùng ảnh đó. Nếu chưa có, notebook dùng ảnh mẫu `flower.jpg` đi kèm scikit-learn.

In [ ]:
print("Họ tên: Lê Nguyễn Anh Du | MSSV: 6451071011")
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.datasets import load_sample_image

HO_TEN, MSSV = "Lê Nguyễn Anh Du", "6451071011"
def in_thong_tin():
    print(f"Họ tên: {HO_TEN} | MSSV: {MSSV}")

in_thong_tin()
print("Đã nhập các thư viện cần thiết.")

## 1) Đọc và hiển thị ảnh gốc

Có thể tải ảnh của bạn lên Colab, đổi tên thành `dog.png` hoặc sửa biến `IMAGE_PATH` bên dưới. Khi tệp đó chưa có, ảnh mẫu sẽ được dùng để notebook chạy ngay.

In [ ]:
in_thong_tin()
IMAGE_PATH = Path("dog.png")
if IMAGE_PATH.is_file():
    image = Image.open(IMAGE_PATH).convert("RGB")
    print(f"Đã đọc ảnh: {IMAGE_PATH}")
else:
    image = Image.fromarray(load_sample_image("flower.jpg")).convert("RGB")
    print("Chưa có dog.png; đã dùng ảnh mẫu flower.jpg của scikit-learn.")
plt.figure(figsize=(8, 5))
plt.imshow(image)
plt.xticks([])
plt.yticks([])
plt.title("Ảnh gốc")
plt.tight_layout()
plt.show()

## 2) Chuyển ảnh thành mảng số và chuẩn hóa màu RGB

In [ ]:
in_thong_tin()
image_array = np.asarray(image)
rows, cols, channels = image_array.shape
if channels != 3:
    raise ValueError("Ảnh phải có ba kênh màu RGB.")
data = image_array.astype("float32") / 255.0
data = data.reshape(rows * cols, 3)
print("Chiều cao:", rows, "| Chiều rộng:", cols, "| Số kênh:", channels)
print("Kích thước dữ liệu pixel để gom cụm:", data.shape)
print("Khoảng giá trị sau chuẩn hóa:", data.min(), "đến", data.max())

## 3) Gom pixel thành 32 màu bằng k-Means

Mỗi pixel là một điểm RGB. Màu của tâm cụm thay cho màu ban đầu của pixel.

In [ ]:
in_thong_tin()
N_COLORS = 32
kmeans = KMeans(n_clusters=N_COLORS, max_iter=300, n_init=3, random_state=42)
labels = kmeans.fit_predict(data)
new_colors = kmeans.cluster_centers_[labels]
image_recolored = np.clip(new_colors.reshape(image_array.shape), 0, 1)
print("Số cụm màu:", N_COLORS)
print("Kích thước ảnh sau tái tạo:", image_recolored.shape)
print("Số màu thực tế sau gom cụm:", len(np.unique(labels)))

## 4) Hiển thị ảnh gốc và ảnh 32 màu cạnh nhau

In [ ]:
in_thong_tin()
fig, axes = plt.subplots(1, 2, figsize=(16, 6), subplot_kw=dict(xticks=[], yticks=[]))
axes[0].imshow(image)
axes[0].set_title("Ảnh gốc", fontsize=16)
axes[1].imshow(image_recolored)
axes[1].set_title("Ảnh 32 màu", fontsize=16)
plt.tight_layout()
plt.show()

## 5) Lưu ảnh và so sánh kích thước tệp

Lưu ảnh mới dưới dạng PNG dùng bảng màu 32 màu để giảm dung lượng. Lưu thêm ảnh gốc dưới dạng PNG để việc so sánh kích thước dùng cùng định dạng tệp.

In [ ]:
in_thong_tin()
original_output = Path("original_for_comparison.png")
compressed_output = Path("compressed_32_colors.png")
image.save(original_output, format="PNG", optimize=True)

# Ảnh P-mode lưu chỉ số màu của từng pixel cùng bảng 32 màu RGB.
centers_uint8 = np.rint(np.clip(kmeans.cluster_centers_, 0, 1) * 255).astype(np.uint8)
indexed_pixels = labels.reshape(rows, cols).astype(np.uint8)
indexed_image = Image.fromarray(indexed_pixels, mode="P")
palette = centers_uint8.reshape(-1).tolist()
indexed_image.putpalette(palette + [0] * (768 - len(palette)))
indexed_image.save(compressed_output, format="PNG", optimize=True)

original_size = original_output.stat().st_size
compressed_size = compressed_output.stat().st_size
print(f"Đã lưu ảnh gốc: {original_output} ({original_size:,} byte)")
print(f"Đã lưu ảnh 32 màu: {compressed_output} ({compressed_size:,} byte)")
print(f"Dung lượng thay đổi: {(1 - compressed_size / original_size) * 100:.2f}%")
print("Có thể tải compressed_32_colors.png từ bảng Files của Colab.")